# 01 · Engine validation — AUROC vs CADD, and cell-type specificity

Validates the **engine** (the ChromBPNet variant score, `|Δ log-counts|`) at ranking
functional regulatory variants above non-functional ones, on the **matched** Kircher
saturation-mutagenesis MPRA benchmark (33,359 SNVs, 29 disease-element assays), against a
**CADD** baseline. This is a *separate* claim from the agent (see notebooks 03–04).

**Run order:** set a GPU runtime, then run top to bottom — the first cell clones the repo.
Full numbers and interpretation live in `RESULTS.md`.

In [ ]:
# --- Setup: clone RegLens from GitHub + install -----------------------------
# No zip upload — clones the public repo fresh, so it's always current. Re-running
# this cell fast-forwards to the latest commit on main.
import os
if not os.path.isdir("/content/RegLens/.git"):
    !git clone --depth 1 https://github.com/kpal002/RegLens.git /content/RegLens
%cd /content/RegLens
!git pull -q --ff-only 2>/dev/null || echo "(using the freshly cloned snapshot)"
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU.'
!pip -q install 'tensorflow>=2.12' pyfaidx typer numpy matplotlib
!pip -q install -e .
!echo "RegLens @ $(git rev-parse --short HEAD) | cwd $(pwd)"

In [ ]:
# --- Reference genome (hg38) + pretrained K562 ChromBPNet model -------------
import glob, os
os.makedirs('/content/ref', exist_ok=True)
%cd /content/ref
# hg38 via the Broad public bucket: resolves on Colab, chr-named, ships its .fai.
!wget -c -O hg38.fa     https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta
!wget -c -O hg38.fa.fai https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai
# ENCODE K562 ATAC ChromBPNet models (5 folds, bias-corrected).
![ -f ENCFF984RAF.tar.gz ] || wget -q -c -O ENCFF984RAF.tar.gz https://www.encodeproject.org/files/ENCFF984RAF/@@download/ENCFF984RAF.tar.gz
!mkdir -p encode_models && tar -xzf ENCFF984RAF.tar.gz -C encode_models
n = len(glob.glob('encode_models/**/*chrombpnet_nobias*.h5', recursive=True))
print(n, 'K562 folds | genome:', os.path.exists('hg38.fa'))
%cd /content/RegLens

In [ ]:
# --- PASS 1: BCL11A subset — a fast, real AUROC ------------------------------
import time
from reglens.validation import load_labeled_variants, evaluate
from reglens.tools.chrombpnet_score import ChromBPNetScorer, load_backend
from reglens.report.render import render_validation

bench = load_labeled_variants('/content/RegLens/data/benchmarks/kircher_mpra_grch38.cadd.tsv')
bcl11a = [v for v in bench if v.source == 'BCL11A']
print(f'BCL11A subset: {len(bcl11a)} variants, {sum(v.label for v in bcl11a)} positive')

scorer = ChromBPNetScorer(load_backend('/content/ref/encode_models', average_rc=True),
                          window_length=2114, model_name='K562-5fold+rc')
t0 = time.time()
rep = evaluate(bcl11a, scorer, genome_path='/content/ref/hg38.fa', progress=True)
print(f'scored in {time.time()-t0:.0f}s'); print(render_validation(rep))

### CADD baseline

The committed benchmark `kircher_mpra_grch38.cadd.tsv` already carries a `cadd` column
(pulled from CADD's pre-scored whole-genome file via `reglens.validation.cadd`), so the
model-vs-CADD comparison needs no extra download.

In [ ]:
# --- PASS 2: full 33k benchmark (GPU) — per-element + overall AUROC ----------
import json, time
from reglens.validation import evaluate_batched
t0 = time.time()
full = evaluate_batched(bench, scorer, genome_path='/content/ref/hg38.fa',
                        chunk_size=2000, progress=True)
print(f'scored {len(bench)} in {time.time()-t0:.0f}s'); print(render_validation(full))
print(json.dumps(full.to_dict(), indent=2)[:1200])

In [ ]:
# --- ROC money-shot: model vs CADD ------------------------------------------
from reglens.report.plot import plot_roc
plot_roc(full, '/content/roc.png')
from IPython.display import Image; Image('/content/roc.png')

In [ ]:
# --- Cell-type specificity: per-element AUROC, hematopoietic highlighted -----
from reglens.report.plot import plot_per_element
from reglens.validation.lineage import render_celltype_specificity
print(render_celltype_specificity(full))
plot_per_element(full, '/content/per_element.png')
from IPython.display import Image; Image('/content/per_element.png')

## Honest reading

A modest overall margin over CADD (~+0.07) whose strength is the **stratified** result:
the K562 (erythroid) model's edge is largest on **hematopoietic** elements and ties CADD
elsewhere — the measurable form of "the right cell-type model matters." The double
dissociation that proves this is causal is in **notebook 02**.